In [ ]:
from RAG.modules import logging

logging.langsmith("Model_RAG")

In [ ]:
from langgraph.graph import StateGraph, START, END
from RAG.types import state
from RAG.llm.model import get_OpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.pydantic_v1 import BaseModel, Field
from langchain.tools import tool

In [17]:
llm = get_OpenAI()
output_parser = StrOutputParser()

In [19]:
from RAG.tools.call_Vehicle_tools import get_brand_list_tool, get_model_list_tool, get_year_list_tool, get_detail_list_tool, get_option_list_tool, HumanRequest, State, Vehicle

In [ ]:
print(f"도구 이름: {get_brand_list_tool.name}")
print(f"도구 설명: {get_brand_list_tool.description}")

In [ ]:
get_brand_list_tool.args_schema.schema()

In [21]:
from langgraph.prebuilt import ToolNode, tools_condition

In [22]:
tools = [get_brand_list_tool, get_model_list_tool, get_year_list_tool, get_detail_list_tool, get_option_list_tool, HumanRequest]

In [23]:
tool_node = ToolNode(tools)

In [ ]:
tool_node

In [25]:
from langchain.globals import set_verbose, set_debug

set_debug(True)
set_verbose(True)

In [26]:
# ToolNode 초기화
tool_node = ToolNode(tools)

In [13]:
# llm_with_tools = llm.bind_tools(tools)

In [14]:
# import os
# from dotenv import load_dotenv
# load_dotenv()

# access_token = os.getenv("CODEF_ACCESS_TOKEN")

# # 요청 헤더 설정
# headers = {
#     "Authorization": f"Bearer {access_token}",
#     "Content-Type": "application/json"
# }

In [1]:
from pydantic import BaseModel
from typing_extensions import Annotated, Sequence, TypedDict
from langchain.schema import BaseMessage

# class headers(TypedDict):
#     Authorization: str
#     Content_Type: str

# class BrandListInput(BaseModel):
#     headers: dict
    
    
class Vehicle(BaseModel):
    brand: str = ""  # 기본값 설정
    model: str = ""
    date: int = 0
    year: str = ""
    detail: str = ""
    option: str = ""

class State(BaseModel):
    headers : dict
    vehicle: Vehicle
    messages: Annotated[Sequence[BaseMessage], "add_messages"]
    ask_human: bool



In [28]:
# LLM 모델 초기화 및 도구 바인딩
llm_with_tools = llm.bind_tools(tools)

In [29]:
from langchain.schema import SystemMessage

In [18]:
def vehicle_search(state: State):
    # 메시지가 비어 있으면 초기화 메시지 추가
    if not state.messages:
        state.messages.append(
            SystemMessage(content="안녕하세요! 차량 정보를 검색하기 위한 대화를 시작합니다. 원하는 제조사를 입력해주세요.")
        )

    def append_message(content: str):
        """사용자 메시지 추가 헬퍼 함수"""
        state.messages.append(BaseMessage(content=content))

    def validate_input(user_input, options, field_name):
        """사용자 입력이 유효한지 검증"""
        if user_input in options:
            return user_input
        else:
            append_message(f"올바른 {field_name}을 선택해주세요: {', '.join(options)}")
            return None

    while state.ask_human:
        print("-- 자동차 정보 수집 --")

        # 현재 상태에서 메시지를 가져옴
        response = llm_with_tools.invoke(state.messages)
        state.messages.append(response)

        # 사용자 응답 처리
        user_input = response.content.strip()
        print(f"사용자 응답: {user_input}")

        # 단계별 처리
        if not state.vehicle.brand:
            # 제조사 선택
            brands = get_brand_list_tool({"headers": state.headers})
            state.vehicle.brand = validate_input(user_input, brands, "제조사")
            if not state.vehicle.brand:
                continue

        elif not state.vehicle.model:
            # 모델 선택
            models = get_model_list_tool(
                {"headers": state.headers, "brand_code": state.vehicle.brand}
            )
            state.vehicle.model = validate_input(user_input, models, "모델")
            if not state.vehicle.model:
                continue

        elif not state.vehicle.year:
            # 연식 선택
            years = get_year_list_tool(
                {"headers": state.headers, "brand_code": state.vehicle.brand, "model_code": state.vehicle.model, "start_date": state.vehicle.date}
            )
            state.vehicle.year = validate_input(user_input, years, "연식")
            if not state.vehicle.year:
                continue

        elif not state.vehicle.detail:
            # 세부 정보 선택
            details = get_detail_list_tool(
                {"headers": state.headers, "brand_code": state.vehicle.brand, "model_code": state.vehicle.model, "year_code": state.vehicle.year}
            )
            state.vehicle.detail = validate_input(user_input, details, "세부 정보")
            if not state.vehicle.detail:
                continue

        elif not state.vehicle.option:
            # 옵션 선택
            options = get_option_list_tool(
                {"headers": state.headers, "brand_code": state.vehicle.brand, "model_code": state.vehicle.model, "year_code": state.vehicle.year, "detail_code": state.vehicle.detail}
            )
            state.vehicle.option = validate_input(user_input, options, "옵션")
            if not state.vehicle.option:
                continue

            state.ask_human = False  # 모든 데이터가 채워지면 대화 종료

    # 최종 선택된 차량 정보 반환
    print("-- 최종 선택된 차량 --")
    print(state.vehicle.json())
    return state

In [30]:
import os
from dotenv import load_dotenv


load_dotenv()

access_token = os.getenv("CODEF_ACCESS_TOKEN")

# 요청 헤더 설정
headers = {
    "Authorization": f"Bearer {access_token}",
    "Content-Type": "application/json"
}

In [ ]:
access_token

In [ ]:
headers

In [ ]:
# 초기 상태 설정
State = State(
    headers=headers,
    vehicle=Vehicle(),
    messages=["차량 조회"],
    ask_human=True
)

In [ ]:
print(type(State.headers))

In [ ]:
state.headers

In [24]:
import requests

In [ ]:
headers = State.headers# headers 추출
url = "https://api.codef.io/v1/kr/car/brand-list"
response = requests.get(url, headers=headers)

In [34]:
import urllib.parse

In [ ]:
decoded_response = urllib.parse.unquote(response.text)
print(decoded_response)

In [ ]:
vehicle_search(state)

In [ ]:
응답함

In [35]:
State : State = {
    "messages": [{"role": "user", "content": "브랜드 조회해줘."}],
    "headers" : headers,
    "vehicle" : Vehicle(),
    "ask_human": False
}

In [ ]:
headers

In [37]:
import requests
import json
import urllib.parse
def get_brand_list_tool(State: dict) -> dict:
    """
    자동차 제조사 목록을 조회하는 함수입니다.

    Args:
        headers (dict): API 호출 시 필요한 헤더 정보.

    Returns:
        dict: 제조사 목록 JSON 데이터.
    """
    headers = State.headers # headers 추출
    url = "https://api.codef.io/v1/kr/car/brand-list"
    response = requests.get(url=url, headers=headers)
    
    if response.status_code == 200:
        try:
            decoded_response = urllib.parse.unquote(response.text)
            return json.loads(decoded_response)
        except json.JSONDecodeError:
            return {"error": "JSON 변환 실패"}
    else:
        return {"error": f"API 요청 실패: {response.status_code}"}

In [ ]:
get_brand_list_tool(State)